In [ ]:
## Orders Fact Table (Olist Dataset)
# This notebook prepares the orders dataset for use as a fact table in an analytical data model.

# The goal is to:
# - Clean and standarsize timestamp fields
# - Engineer delivery performance metrics
# - Filter to valid completed orders
# - Create a reliable fact table ('fact_orders') for downstream analysis

In [ ]:
## Setup for file load

In [ ]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

In [ ]:
## Initial Data Exploration

In [ ]:
orders.head()

In [ ]:
orders.info()

In [ ]:
## Data Cleaning & Validation

In [ ]:
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]

In [ ]:
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

In [ ]:
orders.isna().sum()

In [ ]:
orders['order_status'].value_counts()

In [ ]:
## Grouped by order status to analyze missing values
orders.groupby('order_status')[["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]].apply(lambda x: x.isna().mean())

In [ ]:
## Feature Engineering: Delivery Time
# Delivery time is calculated as the number of days between purchase and customer delivery.
# This metric is used to analyze delivery performance.

In [ ]:
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

In [ ]:
orders["delivery_days"].describe()

In [ ]:
## Outlier & Missing Data Analysis
#  - Identified orders with delivery_days > 100
#  - Investigated missing delivery_days values
#  - Determined that most missing values are expected based on order status

In [ ]:
(orders["delivery_days"] > 100).sum()

In [ ]:
orders[orders["delivery_days"] > 100][["order_id","order_status","delivery_days"]].head(10)

In [ ]:
orders["delivery_days"].isna().sum()

In [ ]:
orders[orders["delivery_days"].isna()]["order_status"].value_counts()

In [ ]:
## Deeper investigation into orders that were delivered but had NA status for delivery days
orders[(orders["delivery_days"].isna()) & (orders["order_status"] == "delivered")][["order_id", "order_purchase_timestamp", "order_delivered_customer_date"]]

In [ ]:
## Final Fact Table: Delivered Orders
#  Filtered the datset to include only:
#    - Orders with status = "delivered"
#    - Orders with valid delivery_days

#  This ensures the dataset only contains complete delivery events for analysis

In [ ]:
fact_orders = orders[(orders["order_status"] == "delivered") & (orders["delivery_days"].notna())]

In [ ]:
fact_orders.isna().sum()

In [ ]:
# The final fact table contains complete delivery records, with only minor missing values in non-critical timestamp fields.

In [ ]:
fact_orders["order_status"].value_counts()

In [ ]:
fact_orders["delivery_days"].describe()

In [ ]:
## Save cleaned Orders Fact Table
fact_orders.to_csv("../data/clean/fact_orders.csv", index=False)